# Vocabulary filtering for topic modelling

### Overview

Input: `checkpoints/df_with_noun_tokens.pkl`, `checkpoints/df_with_term_stats.pkl`, `data/fashion_ontology.csv`

Output: `checkpoints/chosen_vocab.pkl`, `outputs_rlda_preprocessing/stopword_record.csv`

Pipeline:
- **1. Setup:** load checkpoints and ontology
- **2. Diagnostics:** n-gram frequency tables, term-level statistics (DF, ppm, entropy, temporal coverage)
- **3. Vocabulary setting:** apply min frequency thresholds (absolute count), log all removals to stopword record
- **4. Output and audit:** check fashion-term retention, save checkpoint
- **5. Save Checkpoint:** save outputs

### Key variables
- `term_stats`: per-term table of DF, ppm, entropy, temporal coverage, is_fashion
- `stopword_record`: DataFrame logging every removed term with reason & source
- `chosen_vocab`: dict containing the locked vocabulary, count matrix, and token lists
- `CHOSEN_MIN_COUNT`, `CHOSEN_MIN_PPM`: thresholds set by inspection of Section 2 diagnostics


# 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.stats import entropy as scipy_entropy


In [ ]:
# Cell 1: Global variables and parameters

# Paths
CHECKPOINT_DIR = Path("checkpoints")
OUT_DIR        = Path("outputs_rlda_preprocessing")

checkpoint_path = CHECKPOINT_DIR / "df_with_noun_tokens.pkl"
term_stats_path = CHECKPOINT_DIR / "df_with_term_stats.pkl"
ONTOLOGY_CSV = Path("data/fashion_ontology.csv")

## Minimum freq thresholds
MIN_COUNT = 5
MIN_PPM   = 10.0

## Entropy flagging
ENTROPY_TOP_PCT = 0.30
RANDOM_STATE = 42

In [ ]:
# Cell 2: Load noun-token checkpoint & term stats checkpoint
df = pd.read_pickle(checkpoint_path)
print(f"Loaded checkpoint: {df.shape}")
# Sanity check
assert "noun_tokens" in df.columns
assert "noun_text"   in df.columns

term_stats_df = pd.read_pickle(term_stats_path)
print(f"Loaded term stats: {term_stats_df.shape}")
assert "doc_entropy"       in term_stats_df.columns, "Missing doc_entropy"
assert "temporal_coverage" in term_stats_df.columns, "Missing temporal_coverage"
print("Term stats OK")

Loaded checkpoint: (6501, 13)
Loaded term stats: (18511, 7)
Term stats OK


# 2. Diagnostics

In [ ]:
# Cell 3: Load ontology and build term sets

ontology_df = pd.read_csv(ONTOLOGY_CSV)
ontology_df["term"]     = ontology_df["term"].astype(str).str.strip().str.lower()
ontology_df["category"] = ontology_df["category"].astype(str).str.strip().str.lower()
if "n_words" not in ontology_df.columns:
    ontology_df["n_words"] = ontology_df["term"].str.split().str.len()

ONTOLOGY_UNIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 1, "term"])
ONTOLOGY_BIGRAMS  = set(ontology_df.loc[ontology_df["n_words"] == 2, "term"])
ONTOLOGY_TRIGRAMS = set(ontology_df.loc[ontology_df["n_words"] == 3, "term"])
ONTOLOGY_ALL      = ONTOLOGY_UNIGRAMS | ONTOLOGY_BIGRAMS | ONTOLOGY_TRIGRAMS
TERM_TO_CATEGORY  = dict(zip(ontology_df["term"], ontology_df["category"]))

print(
    f"Ontology: {len(ONTOLOGY_UNIGRAMS)} unigrams, "
    f"{len(ONTOLOGY_BIGRAMS)} bigrams, "
    f"{len(ONTOLOGY_TRIGRAMS)} trigrams"
)


Ontology: 683 unigrams, 441 bigrams, 91 trigrams


In [ ]:
# Cell 4: Build n-gram occurrence table, only for diagnostics
# Note: make_ngrams is defined once here and used only in this section.

def make_ngrams(tokens: list[str], n: int) -> list[str]:
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


ngram_rows = []
for _, row in df[["doc_id", "noun_tokens"]].iterrows():
    toks = row["noun_tokens"]
    for term in toks:
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 1})
    for term in make_ngrams(toks, 2):
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 2})
    for term in make_ngrams(toks, 3):
        ngram_rows.append({"doc_id": row["doc_id"], "term": term, "n": 3})

ngram_df = pd.DataFrame(ngram_rows)
print(f"N-gram table: {len(ngram_df):,} rows")
ngram_df.head()


N-gram table: 1,190,385 rows


,doc_id,term,n
0,0,coed,1
1,0,sailor,1
2,0,pant,1
3,0,jacket,1
4,0,simple,1


In [6]:
# Cell 5: Tag each n-gram as fashion-related via ontology set membership
_ontology_by_n = {1: ONTOLOGY_UNIGRAMS, 2: ONTOLOGY_BIGRAMS, 3: ONTOLOGY_TRIGRAMS}

ngram_df["term_lower"]      = ngram_df["term"].str.lower()
ngram_df["is_fashion"]      = ngram_df.apply(
    lambda r: r["term_lower"] in _ontology_by_n.get(r["n"], set()), axis=1
)
ngram_df["fashion_category"] = ngram_df["term_lower"].map(TERM_TO_CATEGORY).fillna("non_fashion")

ngram_df.head()


,doc_id,term,n,term_lower,is_fashion,fashion_category
0,0,coed,1,coed,False,non_fashion
1,0,sailor,1,sailor,True,garment
2,0,pant,1,pant,True,garment
3,0,jacket,1,jacket,True,garment
4,0,simple,1,simple,False,non_fashion


In [7]:
# Cell 6: Corpus-level term frequency tables for manual inspection
term_counts = (
    ngram_df.groupby(["n", "term", "is_fashion", "fashion_category"])
    .size()
    .reset_index(name="corpus_freq")
    .sort_values(["n", "corpus_freq"], ascending=[True, False])
)

top_kept_fashion_terms     = term_counts[term_counts["is_fashion"]].copy()
top_kept_non_fashion_terms = term_counts[~term_counts["is_fashion"]].copy()
matched_ontology_bigrams   = term_counts[(term_counts["n"] == 2) & term_counts["is_fashion"]].copy()
matched_ontology_trigrams  = term_counts[(term_counts["n"] == 3) & term_counts["is_fashion"]].copy()
ambiguous_high_freq_terms  = term_counts[(~term_counts["is_fashion"]) & (term_counts["corpus_freq"] >= 10)].copy()

print(f"Unique unigrams in corpus: {(term_counts.n == 1).sum():,}")
print(f"Fashion unigrams matched:  {((term_counts.n == 1) & term_counts['is_fashion']).sum():,}")
top_kept_non_fashion_terms.head(20)


Unique unigrams in corpus: 18,511
Fashion unigrams matched:  419


,n,term,is_fashion,fashion_category,corpus_freq
1420,1,big,False,non_fashion,1096
14404,1,signature,False,non_fashion,1057
7137,1,hand,False,non_fashion,1034
1617,1,body,False,non_fashion,1017
5569,1,fall,False,non_fashion,980
15480,1,strong,False,non_fashion,944
11243,1,pair,False,non_fashion,902
18433,1,young,False,non_fashion,873
15126,1,spring,False,non_fashion,865
6932,1,great,False,non_fashion,857


In [8]:
# Cell 7: Save baseline EDA tables
term_counts.to_csv(OUT_DIR / "baseline_term_counts.csv", index=False)
top_kept_fashion_terms.to_csv(OUT_DIR / "baseline_top_kept_fashion_terms.csv", index=False)
top_kept_non_fashion_terms.to_csv(OUT_DIR / "baseline_top_kept_non_fashion_terms.csv", index=False)
matched_ontology_bigrams.to_csv(OUT_DIR / "baseline_matched_ontology_bigrams.csv", index=False)
matched_ontology_trigrams.to_csv(OUT_DIR / "baseline_matched_ontology_trigrams.csv", index=False)
ambiguous_high_freq_terms.to_csv(OUT_DIR / "baseline_ambiguous_high_freq_terms.csv", index=False)

print(f"Saved baseline tables to: {OUT_DIR.resolve()}")


Saved baseline tables to: /Users/zoeoggel/Data Science/Thesis/Thesis-Git/outputs_rlda_preprocessing


In [9]:
# Cell 8: Baseline vectorisers (no thresholding), for TF-IDF diagnostic only

docs = df["noun_text"].fillna("").tolist()

base_count_vec = CountVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b")
base_tfidf_vec = TfidfVectorizer(lowercase=False, token_pattern=r"(?u)\b\w+\b")

X_count_base = base_count_vec.fit_transform(docs)
X_tfidf_base = base_tfidf_vec.fit_transform(docs)

count_terms      = np.array(base_count_vec.get_feature_names_out())
total_tokens     = int(X_count_base.sum())

print(f"Baseline count matrix: {X_count_base.shape}")
print(f"Total tokens in corpus: {total_tokens:,}")

Baseline count matrix: (6501, 18511)
Total tokens in corpus: 403,296


In [10]:
# Cell 9: Term-level statistics for manual inspection
term_corpus_freq = np.asarray(X_count_base.sum(axis=0)).ravel()
term_mean_tfidf  = np.asarray(X_tfidf_base.mean(axis=0)).ravel()
term_ppm         = (term_corpus_freq / total_tokens) * 1_000_000

term_stats = pd.DataFrame({
    "term":        count_terms,
    "corpus_freq": term_corpus_freq,
    "ppm":         term_ppm,
    "mean_tfidf":  term_mean_tfidf,
})

# Merge entropy, temporal coverage, burstiness from previous notebook
term_stats = term_stats.merge(
    term_stats_df[["lemma", "doc_freq", "doc_freq_prop", "doc_entropy", "temporal_coverage"]]
    .rename(columns={"lemma": "term"}),
    on="term", how="left"
)

term_stats["is_fashion"]       = term_stats["term"].str.lower().isin(ONTOLOGY_UNIGRAMS)
term_stats["fashion_category"] = term_stats["term"].str.lower().map(TERM_TO_CATEGORY).fillna("non_fashion")

In [11]:
# Cell 10: Inspect max DF proportion
max_df_observed = term_stats["doc_freq_prop"].max()
print(f"Max observed DF proportion: {max_df_observed:.3f}")
print(f"Master term stats table: {term_stats.shape}")
term_stats.sort_values("doc_freq_prop", ascending=False).head(20)

Max observed DF proportion: 0.458
Master term stats table: (18511, 10)


,term,corpus_freq,ppm,mean_tfidf,doc_freq,doc_freq_prop,doc_entropy,temporal_coverage,is_fashion,fashion_category
8413,jacket,4066,10081.924939,0.027178,2976,0.457776,0.900264,16,True,garment
14558,skirt,3892,9650.480044,0.026639,2884,0.443624,0.897298,16,True,garment
2911,coat,3102,7691.621043,0.023256,2308,0.355022,0.871935,16,True,garment
12345,print,3169,7857.752123,0.024361,2253,0.346562,0.866813,16,True,style
8866,leather,2979,7386.634135,0.023942,2072,0.318720,0.856096,16,True,material
18125,white,2527,6265.869237,0.020896,1906,0.293186,0.849564,16,True,color
11294,pant,2172,5385.622471,0.018756,1834,0.282110,0.849032,16,True,garment
14431,silk,2159,5353.388082,0.018972,1723,0.265036,0.839773,16,True,material
14290,short,1930,4785.566929,0.017818,1551,0.238579,0.828277,16,True,garment
7403,high,1497,3711.913830,0.014192,1295,0.199200,0.810375,16,True,garment


In [ ]:
# Cell 11: Check for lower threshold

sweep_counts = [2, 3, 5]
records = []

for min_count in sweep_counts:
    kept = term_stats[term_stats["corpus_freq"] >= min_count]
    kept_fashion = kept[kept["is_fashion"]]
    lost_at_5    = term_stats[
        (term_stats["corpus_freq"] >= min_count) &
        (term_stats["corpus_freq"] < 5) &
        (term_stats["is_fashion"])
    ]
    records.append({
        "min_count":          min_count,
        "vocab_size":         len(kept),
        "fashion_terms":      len(kept_fashion),
        "fashion_gained_vs5": len(lost_at_5),
    })

sweep_df = pd.DataFrame(records)
print("Min-count sweep")
print(sweep_df.to_string(index=False))
print()
print("Fashion terms recovered by lowering from 5 -> 3")
recovered = term_stats[
    (term_stats["corpus_freq"] >= 3) &
    (term_stats["corpus_freq"] < 5) &
    (term_stats["is_fashion"])
].sort_values("corpus_freq", ascending=False)
print(recovered[["term", "corpus_freq", "ppm", "fashion_category"]].to_string(index=False))

Min-count sweep
 min_count  vocab_size  fashion_terms  fashion_gained_vs5
         2       11656            399                  26
         3        9304            389                  16
         5        7083            373                   0

Fashion terms recovered by lowering from 5 -> 3
      term  corpus_freq      ppm fashion_category
broadcloth            4 9.918273         material
     honor            4 9.918273            brand
    pongee            4 9.918273         material
       sea            4 9.918273            brand
 sharkskin            4 9.918273         material
      toga            4 9.918273            brand
 velveteen            4 9.918273         material
   acetate            3 7.438705         material
   acrylic            3 7.438705         material
    henley            3 7.438705          garment
     llama            3 7.438705         material
     lycra            3 7.438705         material
     pique            3 7.438705         material
   

In [14]:
# Cell 12: High-entropy candidate inspection

entropy_threshold = term_stats["doc_entropy"].quantile(1 - ENTROPY_TOP_PCT)
high_entropy_candidates = (
    term_stats[term_stats["doc_entropy"] >= entropy_threshold]
    .sort_values("doc_entropy", ascending=False)
    [["term", "corpus_freq", "ppm", "doc_freq_prop", "doc_entropy", "is_fashion", "fashion_category"]]
)
print(f"High-entropy candidates (top {int(ENTROPY_TOP_PCT*100)}%): {len(high_entropy_candidates)}")
print(f"Of which fashion terms: {high_entropy_candidates['is_fashion'].sum()}")
high_entropy_candidates.head(15)

High-entropy candidates (top 30%): 5785
Of which fashion terms: 359


,term,corpus_freq,ppm,doc_freq_prop,doc_entropy,is_fashion,fashion_category
8413,jacket,4066,10081.924939,0.457776,0.900264,True,garment
14558,skirt,3892,9650.480044,0.443624,0.897298,True,garment
2911,coat,3102,7691.621043,0.355022,0.871935,True,garment
12345,print,3169,7857.752123,0.346562,0.866813,True,style
8866,leather,2979,7386.634135,0.318720,0.856096,True,material
18125,white,2527,6265.869237,0.293186,0.849564,True,color
11294,pant,2172,5385.622471,0.282110,0.849032,True,garment
14431,silk,2159,5353.388082,0.265036,0.839773,True,material
14290,short,1930,4785.566929,0.238579,0.828277,True,garment
7403,high,1497,3711.913830,0.199200,0.810375,True,garment


# 3. Vocabulary setting

In [ ]:
# Cell 14: Set chosen thresholds

# Frequency filter
CHOSEN_MIN_COUNT = 5
CHOSEN_MIN_PPM   = 0


print("Chosen vocabulary parameters:")
print(f"  min_count          = {CHOSEN_MIN_COUNT}")
print(f"  min_ppm            = {CHOSEN_MIN_PPM}")
print(f"  no upper DF filter — max observed DF proportion: {term_stats['doc_freq_prop'].max():.3f}")

Chosen vocabulary parameters:
  min_count          = 5
  min_ppm            = 0
  no upper DF filter — max observed DF proportion: 0.458


In [16]:
# Cell 15: Initialize stopword record

record_columns = [
    "term", "removal_reason", "added_by",
    "corpus_freq", "ppm", "doc_freq_prop", "mean_tfidf",
    "doc_entropy", "temporal_coverage", "is_fashion"
]
stopword_record = pd.DataFrame(columns=record_columns)

def add_to_stopword_record(terms_iter, removal_reason, added_by):
    """Log removed terms to the stopword record with their statistics."""
    global stopword_record
    rows = term_stats[term_stats["term"].isin(set(terms_iter))][
        ["term", "corpus_freq", "ppm", "doc_freq_prop", "mean_tfidf",
         "doc_entropy", "temporal_coverage", "is_fashion"]
    ].copy()
    rows["removal_reason"] = removal_reason
    rows["added_by"]       = added_by
    stopword_record = pd.concat([stopword_record, rows[record_columns]], ignore_index=True)
    print(f"  Record: +{len(rows)} terms ({removal_reason})")


### 3.1. Frequency filter (min count + min ppm)

In [17]:
# Cell 16: Frequency filter

docs = df["noun_text"].fillna("").tolist()

excluded_freq = set(
    term_stats.loc[
        (term_stats["corpus_freq"] <= CHOSEN_MIN_COUNT) |
        (term_stats["ppm"] <= CHOSEN_MIN_PPM),
        "term"
    ].str.lower()
)

surviving = set(term_stats["term"].str.lower()) - excluded_freq

add_to_stopword_record(excluded_freq, removal_reason="below_min_frequency", added_by="frequency_filter")

print(f"Excluded by frequency filter (count<={CHOSEN_MIN_COUNT} OR ppm<={CHOSEN_MIN_PPM}): {len(excluded_freq)}")
print(f"Surviving after frequency filter: {len(surviving)}")

  Record: +12144 terms (below_min_frequency)
Excluded by frequency filter (count<=5 OR ppm<=0): 12144
Surviving after frequency filter: 6367


### 3.2. Final vocabulary assembly

In [18]:
# Cell 17: Vocabulary assembly

kept_terms = surviving

filtered_docs = [
    " ".join(t.lower() for t in toks if t.lower() in kept_terms)
    for toks in df["noun_tokens"]
]

count_vec_final = CountVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    vocabulary={t: i for i, t in enumerate(sorted(kept_terms))},
)
X_count_final       = count_vec_final.fit_transform(filtered_docs)
feature_names_final = np.array(count_vec_final.get_feature_names_out())

token_lists_all = [
    [t.lower() for t in toks if t.lower() in kept_terms]
    for toks in df["noun_tokens"]
]
n_empty_docs = sum(1 for tl in token_lists_all if not tl)
token_lists  = [tl for tl in token_lists_all if tl]

kept_fashion_share = float(np.mean([t in ONTOLOGY_UNIGRAMS for t in kept_terms]))

print(f"Final vocabulary: {len(kept_terms)} terms")
print(f"Empty documents after filtering: {n_empty_docs}")
print(f"Fashion-term retention: {kept_fashion_share:.3f}")

Final vocabulary: 6367 terms
Empty documents after filtering: 0
Fashion-term retention: 0.057


# 4. Output & Inspection

In [19]:
# Cell 18: Fashion-term retention inspection

ontology_in_corpus     = ONTOLOGY_UNIGRAMS & set(term_stats["term"].str.lower())
ontology_removed       = ontology_in_corpus & excluded_freq
ontology_retained      = ontology_in_corpus & kept_terms
fashion_retention_rate = len(ontology_retained) / len(ontology_in_corpus) if ontology_in_corpus else 1.0

print(f"Ontology unigrams in corpus:  {len(ontology_in_corpus)}")
print(f"Ontology terms retained:      {len(ontology_retained)}")
print(f"Ontology terms removed:       {len(ontology_removed)}")
print(f"Fashion retention rate:       {fashion_retention_rate:.3f}")

Ontology unigrams in corpus:  419
Ontology terms retained:      366
Ontology terms removed:       53
Fashion retention rate:       0.874


# 5. Save checkpoint

In [20]:
# Cell 20: Save final chosen vocabulary

chosen_vocab = {
    "kept_terms":         kept_terms,
    "feature_names":      feature_names_final,
    "X_count":            X_count_final,
    "token_lists":        token_lists,
    "n_empty_docs":       n_empty_docs,
    "vocab_size":         len(kept_terms),
    "kept_fashion_share": kept_fashion_share,
    "filtered_docs":      filtered_docs,
}

In [ ]:
# Cell 19: Save vocabulary to checkpoint

checkpoint_out = CHECKPOINT_DIR / "chosen_vocab.pkl"
with open(checkpoint_out, "wb") as f:
    pickle.dump(
        {
            "chosen_vocab":          chosen_vocab,
            "CHOSEN_MIN_COUNT":      CHOSEN_MIN_COUNT,
            "ONTOLOGY_UNIGRAMS":     ONTOLOGY_UNIGRAMS,
            "ONTOLOGY_BIGRAMS":      ONTOLOGY_BIGRAMS,
            "ONTOLOGY_TRIGRAMS":     ONTOLOGY_TRIGRAMS,
            "TERM_TO_CATEGORY":      TERM_TO_CATEGORY,
            "stopword_record":       stopword_record,
            "RANDOM_STATE":          RANDOM_STATE,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

record_out = OUT_DIR / "stopword_record.csv"
stopword_record.to_csv(record_out, index=False)

print(f"Checkpoint saved: {checkpoint_out.resolve()}")
print(f"  vocab_size        = {len(kept_terms)}")
print(f"  fashion_retention = {kept_fashion_share:.3f}")
print(f"  n_empty_docs      = {n_empty_docs}")
print(f"  stopword_record   = {len(stopword_record)} entries → {record_out}")